In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, roc_curve, confusion_matrix, roc_auc_score, classification_report

In [ ]:
df = pd.read_csv('mountains_vs_beaches_preferences.csv')
df.head()

**Inspecting dataset**

In [ ]:
print(df.shape)

In [ ]:
print(df.dtypes)

In [ ]:
df[df.duplicated]

In [ ]:
df.isnull().mean().sort_values()

In [ ]:
df['Preference'].value_counts()

In [ ]:
# the target class is unbalance, accuracy score is not suitable to evaluate the data model

In [ ]:
categorical_features = []
for column in df.columns:
    if df[column].dtype == object:
        print(df[column].value_counts(), '\n')
        categorical_features.append(column)

print(categorical_features)

**Feature Engineering**

One-hot encoding categorical features

In [ ]:
for feature in categorical_features:
    binned = pd.get_dummies(df[feature])
    binned = binned.drop(binned.columns[-1], axis=1) # drop the last dummy variable
    df = df.join(binned)
    df = df.drop(feature, axis=1)

df.head()

Detecting correlated features

In [ ]:
def correlatedFeatures(dataset, threshold):
    """
    Function to detect correlated features

    Input: df, correlation threshold
    Return: Set of correlated features
    """
    correlated_columns = set()
    correlations = dataset.corr()
    for i in range(len(correlations)):
        for j in range(i):
            if abs(correlations.iloc[i,j]) > threshold:
                correlated_columns.add(correlations.columns[i])
    return correlated_columns

In [ ]:
correlatedFeatures(df, 0.8)

In [ ]:
# no highly correlated feature (correlation index > 0.8)

Detect low variance feature

In [ ]:
X = df.drop('Preference', axis=1)
y = df['Preference']

test_size = 0.33
seed = 1

scaler = MinMaxScaler(feature_range=(0,1))
X_ = scaler.fit_transform(X)
X = pd.DataFrame(X_, columns=X.columns)

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)

In [ ]:
sel = VarianceThreshold(threshold=(0.02))
sel.fit(X)

print("Feature selection", sel.get_support())
print("Selected features:", list(X.columns[sel.get_support()]))
print("Removed features:", list(X.columns[~sel.get_support()]))

In [ ]:
# No low variance feature

Select N strongly related features

In [ ]:
kbest = SelectKBest(score_func=f_classif, k=6)
kbest.fit(X, y)

print("Feature selection", kbest.get_support())
print("Feature scores", kbest.scores_)
print("Selected features:", list(X.columns[kbest.get_support()]))
print("Removed features:", list(X.columns[~kbest.get_support()]))

In [ ]:
test_result_df = pd.DataFrame(columns=['N', 'Score'])
n_list = []
score_list = []
for n_best in range(3, 15):
    kbest = SelectKBest(score_func=f_classif, k=n_best)
    kbest.fit(X, y)
    X_ = kbest.transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_, y, test_size=test_size, random_state=seed)
    model = LogisticRegression()
    model.fit(X_train, y_train)
    yhat = model.predict(X_test)
    score = accuracy_score(y_pred=yhat, y_true=y_test)
    # print(str(n_best), ': ', score)
    n_list.append(n_best)
    score_list.append(score)
test_result_df['N'] = pd.Series(n_list)
test_result_df['Score'] = pd.Series(score_list)
plt.plot(test_result_df['N'], test_result_df['Score'])

In [ ]:
# since the performance increase is less obvious after N = 5, first 5 features which have the strongest relationship with the target are chosen

**Building Classification Model**

In [ ]:
def plotRocAuc(model, X, y):
    """ 
    Function to plot the ROC curve and display the AUC

    Input: model, predictors, target
    Return: None
    """
    probabilities = model.predict_proba(X)
    probabilities = probabilities[:, 1]  # keep probabilities for first class only
        
    # Compute the ROC curve
    fpr, tpr, thresholds = roc_curve(y, probabilities)    
    
    # Plot the "dumb model" line
    plt.plot([0, 1], [0, 1], linestyle='--')
    
    # Plot the model line
    plt.plot(fpr, tpr, marker='.')
    plt.text(0.75, 0.25, "AUC: " + str(round(roc_auc_score(y, probabilities), 6)))
    
    # show the plot
    plt.show()

In [ ]:
kbest = SelectKBest(score_func=f_classif, k=5)
kbest.fit(X, y)
print("Selected features:", list(X.columns[kbest.get_support()]), '\n\n')
X_ = kbest.transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_, y, test_size=test_size, random_state=seed)

models = [LogisticRegression(), DecisionTreeClassifier(), KNeighborsClassifier()]
for model in models:
    model.fit(X_train, y_train)
    yhat = model.predict(X_test)
    print("----------------------------------")
    print(type(model).__name__)
    print("----------------------------------")
    print("Confusion matrix:")
    print(confusion_matrix(y_true=y_test, y_pred=yhat))

    print("\nAccuracy:", accuracy_score(y_true=y_test, y_pred=yhat))

    print("\nClassification report:")
    print(classification_report(y_true=y_test, y_pred=yhat))

    print("\nROC / AUC:")
    plotRocAuc(model, X_test, y_test)

In [ ]:
# Logistic Regression is chosen because it has the least false classification in the confusion and highest AUC

Further evaluation with cross validation

In [ ]:
model = LogisticRegression()

kfold = KFold(n_splits=5, shuffle=True, random_state=seed)

kbest = SelectKBest(score_func=f_classif, k=5)
kbest.fit(X, y)
print("Selected features:", list(X.columns[kbest.get_support()]), '\n')
X_ = kbest.transform(X)

# Train and evaluate multiple models using kfolds
results = cross_val_score(model, X_, y, cv=kfold, scoring='roc_auc')
print(results)
print("Mean:", results.mean())
print("Std:", results.std())

In [ ]:
# the model performs accurately and consistently with the average AUC = 0.99998964 with standard deviation < 0.000008

In [ ]:
# the model can be further tested with other datasets to further verify its biasness and comprehensiveness